In [39]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    brier_score_loss,
    log_loss,
    roc_auc_score,
    accuracy_score,
)
from sklearn.calibration import CalibrationDisplay

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
DATA_DIR = '../data/raw/'           # adjust to your Kaggle data path
GENDER = 'M'                 # 'M' for mens, 'W' for womens
VAL_START_SEASON = 2016      # first season used as a validation fold

import logging
logging.getLogger('mlflow').setLevel(logging.ERROR)

from xgboost import XGBClassifier
from sklearn.model_selection import ParameterGrid

pd.set_option('display.max_columns', None)

In [40]:

EXPERIMENT_NAME = f'march_madness_{GENDER.lower()}_2026_v3'

mlflow.set_tracking_uri(f'../mlruns')
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'MLflow experiment: {EXPERIMENT_NAME}')

MLflow experiment: march_madness_m_2026_v3


In [41]:
prefix = GENDER  # 'M' or 'W'

tourney    = pd.read_csv(f'{DATA_DIR}{prefix}NCAATourneyCompactResults.csv')
seeds      = pd.read_csv(f'{DATA_DIR}{prefix}NCAATourneySeeds.csv')
reg_season = pd.read_csv(f'{DATA_DIR}{prefix}RegularSeasonCompactResults.csv')

try:
    reg_detailed = pd.read_csv(f'{DATA_DIR}{prefix}RegularSeasonDetailedResults.csv')
    print(f'Detailed box scores: {reg_detailed.Season.min()}–{reg_detailed.Season.max()}')
except FileNotFoundError:
    reg_detailed = None
    print('No detailed results file — basic features only.')

try:
    massey = pd.read_csv(f'{DATA_DIR}{prefix}MasseyOrdinals.csv')
    print(f'Massey ordinals: {massey.Season.min()}–{massey.Season.max()}')
except FileNotFoundError:
    massey = None
    print('No Massey ordinals file found.')

print(f'Tourney games: {len(tourney)}')
print(f'Seasons: {tourney.Season.min()}–{tourney.Season.max()}')

Detailed box scores: 2003–2026
Massey ordinals: 2003–2026
Tourney games: 2585
Seasons: 1985–2025


## Seeds

In [42]:
# ── 4a. Parse seed number from seed string (e.g. 'W01' → 1) ──────────────────
def parse_seed(seed_str):
    return int(''.join(filter(str.isdigit, str(seed_str))))

seeds['SeedNum'] = seeds['Seed'].apply(parse_seed)
seed_lookup = seeds.set_index(['Season', 'TeamID'])['SeedNum'].to_dict()

## Elo Rating

In [43]:
# ── 4b. Elo ratings (carries across seasons with mean reversion) ──────────────
def compute_elo(results_df, k=32, initial=1500, carry_factor=0.75, use_mov=True):
    elo = {}
    season_end = {}

    for season, grp in results_df.sort_values(['Season', 'DayNum']).groupby('Season'):
        elo = {
            team: carry_factor * rating + (1 - carry_factor) * initial
            for team, rating in elo.items()
        }

        for _, row in grp.iterrows():
            w, l = row['WTeamID'], row['LTeamID']
            r_w = elo.get(w, initial)
            r_l = elo.get(l, initial)

            # Home court adjustment
            loc = row.get('WLoc', 'N')
            hca = 100
            r_w_adj = r_w + (hca if loc == 'H' else -hca if loc == 'A' else 0)

            exp_w = 1 / (1 + 10 ** ((r_l - r_w_adj) / 400))
            exp_l = 1 - exp_w

            # MOV adjusted K
            if use_mov and 'WScore' in row.index:
                mov = row['WScore'] - row['LScore']
                elo_diff = abs(r_w_adj - r_l)
                mov_mult = np.log(abs(mov) + 1.0)
                elo_mult = 2.2 / ((elo_diff * 0.001) + 2.2)
                k_adj = k * mov_mult * elo_mult
            else:
                k_adj = k

            elo[w] = r_w + k_adj * (1 - exp_w)
            elo[l] = r_l + k_adj * (0 - exp_l)

        for team, rating in elo.items():
            season_end[(season, team)] = rating

    return season_end

print('Computing Elo ratings...')
elo_ratings = compute_elo(reg_season, k=20, carry_factor=0.75)
print(f'Elo ratings computed for {len(elo_ratings)} (season, team) pairs.')

Computing Elo ratings...
Elo ratings computed for 14206 (season, team) pairs.


## Season and Last-N Advanced Metrics

In [44]:
# ── 4c. Last-N form features from detailed box scores ────────────────────────
def compute_form_features(detailed_df, n_games=10):
    if detailed_df is None:
        return pd.DataFrame()

    records = []
    for _, row in detailed_df.iterrows():
        season = row['Season']
        day    = row['DayNum']

        w_poss = row['WFGA'] - row['WOR'] + row['WTO'] + 0.44 * row['WFTA']
        l_poss = row['LFGA'] - row['LOR'] + row['LTO'] + 0.44 * row['LFTA']
        poss   = (w_poss + l_poss) / 2 if (w_poss + l_poss) > 0 else 1

        w_off_eff = (row['WScore'] / poss) * 100
        l_off_eff = (row['LScore'] / poss) * 100

        records.append({
            'Season': season, 'DayNum': day,
            'TeamID': row['WTeamID'], 'Win': 1,
            'Margin': row['WScore'] - row['LScore'],
            'OffEff': w_off_eff, 'DefEff': l_off_eff,
            'Pace': poss
        })
        records.append({
            'Season': season, 'DayNum': day,
            'TeamID': row['LTeamID'], 'Win': 0,
            'Margin': row['LScore'] - row['WScore'],
            'OffEff': l_off_eff, 'DefEff': w_off_eff,
            'Pace': poss
        })

    game_df = pd.DataFrame(records).sort_values(['Season', 'TeamID', 'DayNum'])

    form_rows = []
    for (season, team), grp in game_df.groupby(['Season', 'TeamID']):
        last_n = grp.tail(n_games)

        # ── Season-long stats ────────────────────────────────────────────────
        season_row = {
            'Season': season,
            'TeamID': team,
            # Season long
            'Season_WinRate':   grp['Win'].mean(),
            'Season_AvgMargin': grp['Margin'].mean(),
            'Season_AvgOffEff': grp['OffEff'].mean(),
            'Season_AvgDefEff': grp['DefEff'].mean(),
            'Season_EffMargin': grp['OffEff'].mean() - grp['DefEff'].mean(),
            'Season_AvgPace':   grp['Pace'].mean(),
            'Season_NumGames':  len(grp),
            # Last N
            f'Last{n_games}_WinRate':   last_n['Win'].mean(),
            f'Last{n_games}_AvgMargin': last_n['Margin'].mean(),
            f'Last{n_games}_AvgOffEff': last_n['OffEff'].mean(),
            f'Last{n_games}_AvgDefEff': last_n['DefEff'].mean(),
            f'Last{n_games}_EffMargin': last_n['OffEff'].mean() - last_n['DefEff'].mean(),
            f'Last{n_games}_AvgPace':   last_n['Pace'].mean(),
            # Trend — last N minus season average, captures peaking/declining
            'Trend_EffMargin': (last_n['OffEff'].mean() - last_n['DefEff'].mean()) - 
                               (grp['OffEff'].mean() - grp['DefEff'].mean()),
            'Trend_WinRate':   last_n['Win'].mean() - grp['Win'].mean(),
        }
        form_rows.append(season_row)

    return pd.DataFrame(form_rows).set_index(['Season', 'TeamID'])

print('Computing form features...')
form_df = compute_form_features(reg_detailed, n_games=10)
if not form_df.empty:
    print(f'Form features: {form_df.shape}')
else:
    print('No form features (detailed data unavailable).')

Computing form features...
Form features: (8346, 15)


## Massey Ordinals

In [46]:
# ── 4d. Massey ordinal snapshot (as of Selection Sunday ~day 133) ─────────────
def get_massey_snapshot(massey_df, system='POM', day_cutoff=133):
    """
    Pull each team's most recent ranking before Selection Sunday.
    Defaults to KenPom (POM).
    """
    if massey_df is None:
        return {}

    filtered = massey_df[massey_df['RankingDayNum'] <= day_cutoff]

    if system in filtered['SystemName'].unique():
        filtered = filtered[filtered['SystemName'] == system]
    else:
        print(f'System {system} not found — using all systems.')

    latest = (
        filtered.sort_values('RankingDayNum')
        .groupby(['Season', 'TeamID'])
        .last()
        .reset_index()
    )

    return latest.set_index(['Season', 'TeamID'])['OrdinalRank'].to_dict()

massey_lookup = get_massey_snapshot(massey, system='POM')
print(f'Massey lookup entries: {len(massey_lookup)}')

Massey lookup entries: 8352


In [47]:
def get_consensus_ordinals(massey_df, systems=['POM', 'SAG', 'NET', 'MOR', 'TRK', 'WIL'], day_cutoff=133):
    """
    Pulls the latest ranking for high-signal systems before Selection Sunday.
    Returns a dictionary keyed by (Season, TeamID) containing median, mean, min, max, std, and IQR.
    """
    if massey_df is None or massey_df.empty:
        return {}

    # 1. Filter up to the day cutoff and only keep our trusted systems
    filtered = massey_df[
        (massey_df['RankingDayNum'] <= day_cutoff) & 
        (massey_df['SystemName'].isin(systems))
    ]

    # 2. Get the *latest* ranking per system just in case a system updated on day 130 and 133
    latest_per_system = (
        filtered.sort_values('RankingDayNum')
        .groupby(['Season', 'TeamID', 'SystemName'])
        .last()
        .reset_index()
    )

    # 3. Custom IQR function
    def iqr(x):
        return x.quantile(0.75) - x.quantile(0.25)

    # 4. Group by Season and Team to calculate consensus stats
    consensus = latest_per_system.groupby(['Season', 'TeamID'])['OrdinalRank'].agg([
        'median', 'mean', 'min', 'max', 'std', iqr
    ]).reset_index()

    # 5. Rename columns for clarity
    consensus = consensus.rename(columns={
        'median': 'RankMedian', 
        'mean': 'RankMean',
        'min': 'RankMin',
        'max': 'RankMax',
        'std': 'RankStd',
        'iqr': 'RankIQR'
    })
    
    # Fill any NaN standard deviations with 0 (happens if a team only has 1 ranking system)
    consensus['RankStd'] = consensus['RankStd'].fillna(0)

    # 6. Convert to dictionary with (Season, TeamID) as the index key
    # orient='index' makes the value a dictionary of the row's features
    lookup_dict = consensus.set_index(['Season', 'TeamID']).to_dict(orient='index')
    
    return lookup_dict

consensus_df = get_consensus_ordinals(massey)
print(f'Consensus lookup entries: {len(consensus_df)}')
print(consensus_df[(2026, 1181)])  # e.g., Duke in 2026

Consensus lookup entries: 8355
{'RankMedian': 1.0, 'RankMean': 1.4, 'RankMin': 1, 'RankMax': 2, 'RankStd': 0.5477225575051662, 'RankIQR': 1.0}


## Coach Performance

In [48]:
coaches = pd.read_csv(f'{DATA_DIR}{prefix}TeamCoaches.csv')
tourney_compact = pd.read_csv(f'{DATA_DIR}{prefix}NCAATourneyCompactResults.csv')

def compute_coach_features(coaches_df, tourney_df):
    # coaches_df already has one row per (Season, TeamID, CoachName)
    # no need to expand — just deduplicate to be safe
    coach_df = coaches_df[['Season', 'TeamID', 'CoachName']].drop_duplicates(
        ['Season', 'TeamID']  # keep first coach if mid-season change
    )

    w_games = tourney_df[['Season', 'WTeamID']].copy()
    w_games.columns = ['Season', 'TeamID']
    w_games['TourneyWin'] = 1

    l_games = tourney_df[['Season', 'LTeamID']].copy()
    l_games.columns = ['Season', 'TeamID']
    l_games['TourneyWin'] = 0

    tourney_games = pd.concat([w_games, l_games])
    tourney_games = tourney_games.merge(coach_df, on=['Season', 'TeamID'], how='left')

    coach_stats = []
    for coach, coach_group in tourney_games.groupby('CoachName'):
        cumulative_wins        = 0
        cumulative_games       = 0
        cumulative_appearances = 0

        for season, season_group in coach_group.groupby('Season'):
            coach_stats.append({
                'Season':               season,
                'CoachName':            coach,
                'Coach_TourneyApps':    cumulative_appearances,
                'Coach_TourneyWins':    cumulative_wins,
                'Coach_TourneyGames':   cumulative_games,
                'Coach_TourneyWinRate': cumulative_wins / cumulative_games
                                        if cumulative_games > 0 else np.nan,
            })
            cumulative_wins        += season_group['TourneyWin'].sum()
            cumulative_games       += len(season_group)
            cumulative_appearances += 1

    coach_stats_df = pd.DataFrame(coach_stats)
    result = coach_df.merge(coach_stats_df, on=['Season', 'CoachName'], how='left')
    return result.set_index(['Season', 'TeamID'])

print('Computing coach features...')
coach_df = compute_coach_features(coaches, tourney)
print(f'Coach features: {coach_df.shape}')

Computing coach features...
Coach features: (13763, 5)


In [49]:
coach_features

['Coach_TourneyApps_Diff',
 'Coach_TourneyWins_Diff',
 'Coach_TourneyWinRate_Diff',
 'Coach_TourneyGames_Diff']

## Four Factors (Shooting, Turnovers, Rebounding, Free Throws)

In [50]:
def compute_four_factors(detailed_df, n_games=10):
    if detailed_df is None:
        return pd.DataFrame()

    records = []
    for _, row in detailed_df.iterrows():
        season = row['Season']
        day    = row['DayNum']

        # Possessions
        w_poss = row['WFGA'] - row['WOR'] + row['WTO'] + 0.44 * row['WFTA']
        l_poss = row['LFGA'] - row['LOR'] + row['LTO'] + 0.44 * row['LFTA']

        # Effective FG% = (FGM + 0.5 * 3PM) / FGA
        w_efg  = (row['WFGM'] + 0.5 * row['WFGM3']) / row['WFGA'] if row['WFGA'] > 0 else 0.5
        l_efg  = (row['LFGM'] + 0.5 * row['LFGM3']) / row['LFGA'] if row['LFGA'] > 0 else 0.5

        # Turnover rate = TO / possessions
        w_tor  = row['WTO'] / w_poss if w_poss > 0 else 0.15
        l_tor  = row['LTO'] / l_poss if l_poss > 0 else 0.15

        # Offensive rebound rate = ORB / (ORB + opp DRB)
        w_orb  = row['WOR'] / (row['WOR'] + row['LDR']) if (row['WOR'] + row['LDR']) > 0 else 0.3
        l_orb  = row['LOR'] / (row['LOR'] + row['WDR']) if (row['LOR'] + row['WDR']) > 0 else 0.3

        # Free throw rate = FTA / FGA
        w_ftr  = row['WFTA'] / row['WFGA'] if row['WFGA'] > 0 else 0.3
        l_ftr  = row['LFTA'] / row['LFGA'] if row['LFGA'] > 0 else 0.3

        # 3-point rate = FGA3 / FGA
        w_3pr  = row['WFGA3'] / row['WFGA'] if row['WFGA'] > 0 else 0.3
        l_3pr  = row['LFGA3'] / row['LFGA'] if row['LFGA'] > 0 else 0.3

        # 3-point % 
        w_3p   = row['WFGM3'] / row['WFGA3'] if row['WFGA3'] > 0 else 0.33
        l_3p   = row['LFGM3'] / row['LFGA3'] if row['LFGA3'] > 0 else 0.33

        records.append({
            'Season': season, 'DayNum': day,
            'TeamID': row['WTeamID'],
            'EFG': w_efg, 'EFGD': l_efg,
            'TOR': w_tor, 'TORD': l_tor,
            'ORB': w_orb, 'DRB': 1 - l_orb,
            'FTR': w_ftr, 'FTRD': l_ftr,
            'ThreePR': w_3pr, 'ThreePRD': l_3pr,
            'ThreeP': w_3p,  'ThreePD': l_3p,
        })
        records.append({
            'Season': season, 'DayNum': day,
            'TeamID': row['LTeamID'],
            'EFG': l_efg, 'EFGD': w_efg,
            'TOR': l_tor, 'TORD': w_tor,
            'ORB': l_orb, 'DRB': 1 - w_orb,
            'FTR': l_ftr, 'FTRD': w_ftr,
            'ThreePR': l_3pr, 'ThreePRD': w_3pr,
            'ThreeP': l_3p,  'ThreePD': w_3p,
        })

    game_df = pd.DataFrame(records).sort_values(['Season', 'TeamID', 'DayNum'])

    ff_rows = []
    for (season, team), grp in game_df.groupby(['Season', 'TeamID']):
        last_n = grp.tail(n_games)
        ff_rows.append({
            'Season': season, 'TeamID': team,
            # Season averages
            'Season_EFG':    grp['EFG'].mean(),
            'Season_EFGD':   grp['EFGD'].mean(),
            'Season_TOR':    grp['TOR'].mean(),
            'Season_TORD':   grp['TORD'].mean(),
            'Season_ORB':    grp['ORB'].mean(),
            'Season_DRB':    grp['DRB'].mean(),
            'Season_FTR':    grp['FTR'].mean(),
            'Season_FTRD':   grp['FTRD'].mean(),
            'Season_3PR':    grp['ThreePR'].mean(),
            'Season_3PRD':   grp['ThreePRD'].mean(),
            'Season_3P':     grp['ThreeP'].mean(),
            'Season_3PD':    grp['ThreePD'].mean(),
            # Last N averages
            f'Last{n_games}_EFG':  last_n['EFG'].mean(),
            f'Last{n_games}_EFGD': last_n['EFGD'].mean(),
            f'Last{n_games}_TOR':  last_n['TOR'].mean(),
            f'Last{n_games}_TORD': last_n['TORD'].mean(),
            f'Last{n_games}_ORB':  last_n['ORB'].mean(),
            f'Last{n_games}_3PR':  last_n['ThreePR'].mean(),
            f'Last{n_games}_3P':   last_n['ThreeP'].mean(),
        })

    return pd.DataFrame(ff_rows).set_index(['Season', 'TeamID'])

print('Computing four factors features...')
ff_df = compute_four_factors(reg_detailed, n_games=10)
print(f'Four factors features: {ff_df.shape}')

Computing four factors features...
Four factors features: (8346, 19)


In [51]:
# Compute opponent win rate lookup — used inside compute_sos_features
win_counts = reg_season.groupby(['Season', 'WTeamID']).size().reset_index()
win_counts.columns = ['Season', 'TeamID', 'Wins']

game_counts = pd.concat([
    reg_season[['Season', 'WTeamID']].rename(columns={'WTeamID': 'TeamID'}),
    reg_season[['Season', 'LTeamID']].rename(columns={'LTeamID': 'TeamID'})
]).groupby(['Season', 'TeamID']).size().reset_index()
game_counts.columns = ['Season', 'TeamID', 'Games']

win_rates = win_counts.merge(game_counts, on=['Season', 'TeamID'])
win_rates['WinRate'] = win_rates['Wins'] / win_rates['Games']
win_rate_lookup = win_rates.set_index(['Season', 'TeamID'])['WinRate'].to_dict()

print(f'Win rate lookup: {len(win_rate_lookup)} entries')

Win rate lookup: 13736 entries


## Strength of Schedule

In [52]:
def compute_sos_features(results_df, elo_ratings, win_rate_lookup, detailed_df=None, n_games=10):
    """
    Compute strength of schedule features for each (season, team).
    
    Two versions:
    - Season SOS: average opponent Elo and efficiency margin across all regular season games
    - Last-N SOS: same but only for the last n games
    
    Returns DataFrame indexed by (Season, TeamID).
    """
    records = []
    for _, row in results_df.sort_values(['Season', 'DayNum']).iterrows():
        season = row['Season']
        w_team, l_team = row['WTeamID'], row['LTeamID']
        w_elo = elo_ratings.get((season, w_team), 1500)
        l_elo = elo_ratings.get((season, l_team), 1500)

        records.append({
            'Season': season, 'DayNum': row['DayNum'],
            'TeamID': w_team, 'OppElo': l_elo,
            'OppWinRate': win_rate_lookup.get((season, l_team), 0.5),
        })
        records.append({
            'Season': season, 'DayNum': row['DayNum'],
            'TeamID': l_team, 'OppElo': w_elo,
            'OppWinRate': win_rate_lookup.get((season, w_team), 0.5),
        })

    opp_df = pd.DataFrame(records).sort_values(['Season', 'TeamID', 'DayNum'])

    # If detailed box scores available, also compute opponent efficiency margin
    if detailed_df is not None:
        # Reuse the game_df logic from compute_form_features
        eff_records = []
        for _, row in detailed_df.iterrows():
            season = row['Season']
            day    = row['DayNum']

            w_poss = row['WFGA'] - row['WOR'] + row['WTO'] + 0.44 * row['WFTA']
            l_poss = row['LFGA'] - row['LOR'] + row['LTO'] + 0.44 * row['LFTA']
            poss   = (w_poss + l_poss) / 2 if (w_poss + l_poss) > 0 else 1

            w_eff_margin = ((row['WScore'] - row['LScore']) / poss) * 100
            l_eff_margin = -w_eff_margin

            eff_records.append({
                'Season': season, 'DayNum': day,
                'TeamID': row['WTeamID'], 'OppEffMargin': l_eff_margin
            })
            eff_records.append({
                'Season': season, 'DayNum': day,
                'TeamID': row['LTeamID'], 'OppEffMargin': w_eff_margin
            })

        eff_df = pd.DataFrame(eff_records).sort_values(['Season', 'TeamID', 'DayNum'])
        opp_df = opp_df.merge(eff_df, on=['Season', 'DayNum', 'TeamID'], how='left')
    else:
        opp_df['OppEffMargin'] = np.nan

    # Compute season and last-N SOS per team
    sos_rows = []
    for (season, team), grp in opp_df.groupby(['Season', 'TeamID']):
        last_n = grp.tail(n_games)
        row = {
            'Season': season,
            'TeamID': team,
            # Season SOS
            'SOS_Season_AvgOppElo':       grp['OppElo'].mean(),
            'SOS_Season_AvgOppEffMargin': grp['OppEffMargin'].mean(),
            'SOS_Season_AvgOppWinRate':   grp['OppWinRate'].mean(),   # ← add this
            # Last-N SOS
            f'SOS_Last{n_games}_AvgOppElo':       last_n['OppElo'].mean(),
            f'SOS_Last{n_games}_AvgOppEffMargin': last_n['OppEffMargin'].mean(),
            f'SOS_Last{n_games}_AvgOppWinRate':   last_n['OppWinRate'].mean(),  # ← and this
        }
        sos_rows.append(row)

    return pd.DataFrame(sos_rows).set_index(['Season', 'TeamID'])


print('Computing SOS features...')
sos_df = compute_sos_features(reg_season, elo_ratings, win_rate_lookup, reg_detailed, n_games=10)
print(f'SOS features: {sos_df.shape}')

Computing SOS features...
SOS features: (13753, 6)


## Build Matchup Features

In [56]:
def build_matchup_features(tourney_df, elo_ratings, seed_lookup, form_df, massey_lookup, coach_df, sos_df, ff_df):
    """
    Standardized feature builder. 
    Accepts DataFrames for form, coach, sos, and ff data.
    """
    rows = []
    
    # Default dict for missing Massey data
    default_massey = {
        'RankMedian': np.nan, 'RankMean': np.nan, 
        'RankMin': np.nan, 'RankMax': np.nan, 
        'RankStd': np.nan, 'RankIQR': np.nan
    }

    for _, game in tourney_df.iterrows():
        season = int(game['Season'])
        # Handle both training data (WTeamID/LTeamID) and sub data (WTeamID/LTeamID as TeamA/TeamB)
        t1, t2 = int(game['WTeamID']), int(game['LTeamID'])

        # Logic to ensure Team A is always the lower TeamID for symmetry
        if t1 < t2:
            t_a, t_b = t1, t2
            label = 1 if 'Label' not in game or game.get('Label', 1) == 1 else 0 
        else:
            t_a, t_b = t2, t1
            label = 0 if 'Label' not in game or game.get('Label', 1) == 1 else 1

        # 1. Core Features
        seed_a = seed_lookup.get((season, t_a), 16)
        seed_b = seed_lookup.get((season, t_b), 16)
        elo_a  = elo_ratings.get((season, t_a), 1500)
        elo_b  = elo_ratings.get((season, t_b), 1500)
        
        m_a = massey_lookup.get((season, t_a), default_massey)
        m_b = massey_lookup.get((season, t_b), default_massey)

        row = {
            'Season': season, 'TeamA': t_a, 'TeamB': t_b, 'Label': label,
            'SeedA': seed_a, 'SeedB': seed_b, 'SeedDiff': seed_a - seed_b,
            'EloA': elo_a,   'EloB': elo_b,   'EloDiff':  elo_a - elo_b,
            'MasseyMedianDiff': m_b['RankMedian'] - m_a['RankMedian'],
            'MasseyMeanDiff': m_b['RankMean'] - m_a['RankMean'],
            'MasseyMinDiff': m_b['RankMin'] - m_a['RankMin'],
            'MasseyMaxDiff': m_b['RankMax'] - m_a['RankMax']
        }

# 2. Dynamic Feature Adder (Fixed to prevent double _Diff)
        def add_stats(target_row, df, prefix):
            if df is not None and hasattr(df, 'columns'):
                for col in df.columns:
                    if df[col].dtype == 'object':
                        continue
                    # If the column already ends in _Diff, don't add it again
                    feature_name = col if col.endswith('_Diff') else f'{col}_Diff'
                    
                    val_a = df.loc[(season, t_a), col] if (season, t_a) in df.index else np.nan
                    val_b = df.loc[(season, t_b), col] if (season, t_b) in df.index else np.nan
                    
                    target_row[feature_name] = val_a - val_b if not (np.isnan(val_a) or np.isnan(val_b)) else np.nan

        # Process the dataframes
        add_stats(row, form_df, "Form")
        add_stats(row, sos_df, "SOS")
        add_stats(row, coach_df, "Coach")
        add_stats(row, ff_df, "FF")

        rows.append(row)

    return pd.DataFrame(rows)

# Create the training features
matchups = build_matchup_features(
    tourney, 
    elo_ratings, 
    seed_lookup, 
    form_df,    # Passed as DataFrame
    consensus_df, 
    coach_df,   # Passed as DataFrame
    sos_df,     # Passed as DataFrame
    ff_df       # Passed as DataFrame
)

print(f'Matchup features: {matchups.shape}')
matchups.head()

Matchup features: (2585, 58)


,Season,TeamA,TeamB,Label,SeedA,SeedB,SeedDiff,EloA,EloB,EloDiff,MasseyMedianDiff,MasseyMeanDiff,MasseyMinDiff,MasseyMaxDiff,Season_WinRate_Diff,Season_AvgMargin_Diff,Season_AvgOffEff_Diff,Season_AvgDefEff_Diff,Season_EffMargin_Diff,Season_AvgPace_Diff,Season_NumGames_Diff,Last10_WinRate_Diff,Last10_AvgMargin_Diff,Last10_AvgOffEff_Diff,Last10_AvgDefEff_Diff,Last10_EffMargin_Diff,Last10_AvgPace_Diff,Trend_EffMargin_Diff,Trend_WinRate_Diff,SOS_Season_AvgOppElo_Diff,SOS_Season_AvgOppEffMargin_Diff,SOS_Season_AvgOppWinRate_Diff,SOS_Last10_AvgOppElo_Diff,SOS_Last10_AvgOppEffMargin_Diff,SOS_Last10_AvgOppWinRate_Diff,Coach_TourneyApps_Diff,Coach_TourneyWins_Diff,Coach_TourneyGames_Diff,Coach_TourneyWinRate_Diff,Season_EFG_Diff,Season_EFGD_Diff,Season_TOR_Diff,Season_TORD_Diff,Season_ORB_Diff,Season_DRB_Diff,Season_FTR_Diff,Season_FTRD_Diff,Season_3PR_Diff,Season_3PRD_Diff,Season_3P_Diff,Season_3PD_Diff,Last10_EFG_Diff,Last10_EFGD_Diff,Last10_TOR_Diff,Last10_TORD_Diff,Last10_ORB_Diff,Last10_3PR_Diff,Last10_3P_Diff
0,1985,1116,1234,1,9,8,1,1647.655692,1657.292812,-9.637120,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.212092,NaN,0.020862,-55.191161,NaN,-0.037698,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1985,1120,1345,1,11,6,5,1613.851268,1648.234155,-34.382887,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-15.920722,NaN,0.002403,-60.229977,NaN,-0.049200,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1985,1207,1250,1,1,16,-15,1885.571632,1403.208004,482.363628,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,131.124751,NaN,0.131907,191.691916,NaN,0.204564,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1985,1229,1425,1,9,8,1,1618.460596,1617.619647,0.840950,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-18.982239,NaN,-0.033687,-9.484906,NaN,-0.040418,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1985,1242,1325,1,3,14,-11,1680.970644,1665.631026,15.339618,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.991989,NaN,0.065588,71.085755,NaN,0.063464,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
def augment_matchups(df):
    """
    Duplicates every row in the matchup dataframe, swapping Team A and Team B,
    flipping the label, and negating all 'Diff' columns to create a perfectly 
    balanced dataset.
    """    
    # Create a copy for the flipped rows
    swapped_df = df.copy()
    
    # 1. Flip the Label (1 becomes 0, 0 becomes 1)
    swapped_df['Label'] = 1 - swapped_df['Label']
    
    # 2. Dynamically find all A and B column pairs
    # This looks for 'TeamA', 'EloA', 'Coach_TourneyWins_A', etc.
    cols_A = [col for col in df.columns if col.endswith('A')]
    cols_B = [col[:-1] + 'B' for col in cols_A]
    
    # Filter to only keep pairs where both A and B versions actually exist
    valid_pairs = [(a, b) for a, b in zip(cols_A, cols_B) if b in df.columns]
    valid_A = [a for a, b in valid_pairs]
    valid_B = [b for a, b in valid_pairs]
    
    # Swap the data between the A columns and B columns
    swapped_df[valid_A + valid_B] = df[valid_B + valid_A].values
    
    # 3. Dynamically find and negate all Difference columns
    # If Diff was (A - B), from the new perspective it must be (B - A), which is -(A - B)
    diff_cols = [col for col in df.columns if 'Diff' in col]
    swapped_df[diff_cols] = -swapped_df[diff_cols]
    
    # 4. Combine the original and swapped dataframes
    combined_df = pd.concat([df, swapped_df], axis=0, ignore_index=True)
    
    # 5. Shuffle the dataset so 1s and 0s are mixed (good for CV and batch training)
    combined_df = combined_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    
    return combined_df

# --- How to use it ---
# matchups = build_matchup_features(...)
print(f"Before augmentation: {matchups.shape[0]} rows, Label mean: {matchups['Label'].mean():.3f}")
balanced_matchups = augment_matchups(matchups)
print(f"After augmentation: {balanced_matchups.shape[0]} rows, Label mean: {balanced_matchups['Label'].mean():.3f}")

Before augmentation: 2585 rows, Label mean: 0.512
After augmentation: 5170 rows, Label mean: 0.500


## Walk Forward, Leave One Out, Cross Validation

In [58]:
def walk_forward_cv(matchups_df, feature_cols, val_start=VAL_START_SEASON,
                    impute=True, season_decay=None):
    SKIP_SEASONS = {2020}
    seasons     = sorted(matchups_df['Season'].unique())
    val_seasons = [s for s in seasons if s >= val_start and s not in SKIP_SEASONS]

    folds = []
    for val_season in val_seasons:
        train_raw = matchups_df[matchups_df['Season'] <  val_season]
        train = augment_matchups(train_raw) 
        val   = matchups_df[matchups_df['Season'] == val_season]

        if len(train) == 0 or len(val) == 0:
            continue

        y_train = train['Label']
        y_val   = val['Label']

        if impute:
            train_median = train[feature_cols].median()
            X_train = train[feature_cols].fillna(train_median)
            X_val   = val[feature_cols].fillna(train_median)
        else:
            X_train = train[feature_cols]
            X_val   = val[feature_cols]

        # Sample weights — exponential decay by season
        if season_decay is not None:
            max_season   = train['Season'].max()
            sample_weights = season_decay ** (max_season - train['Season'])
            sample_weights = sample_weights / sample_weights.sum() * len(train)
        else:
            sample_weights = None

        folds.append((val_season, X_train, y_train, X_val, y_val, sample_weights))

    return folds

## Compute Metrics

In [19]:
def compute_metrics(y_true, y_pred_proba):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred_proba)

    return {
        'brier_score': brier_score_loss(y_true, y_pred),
        'log_loss':    log_loss(y_true, y_pred),
        'roc_auc':     roc_auc_score(y_true, y_pred),
        'accuracy':    accuracy_score(y_true, (y_pred >= 0.5).astype(int)),
        'mean_pred':   y_pred.mean(),
        'n_games':     len(y_true),
    }


def plot_calibration(all_y_true, all_y_pred, run_name):
    fig, ax = plt.subplots(figsize=(6, 5))
    CalibrationDisplay.from_predictions(
        np.concatenate(all_y_true),
        np.concatenate(all_y_pred),
        n_bins=10, ax=ax, name=run_name
    )
    ax.set_title(f'Calibration — {run_name}')
    plt.tight_layout()
    return fig

## Run Experiment

In [20]:
def run_experiment(run_name, model, feature_cols, matchups_df,
                   tags=None, val_start=VAL_START_SEASON, scale=True, impute=True, season_decay=None):
    steps = []
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    pipeline = Pipeline(steps)

    folds = walk_forward_cv(matchups_df, feature_cols, val_start=val_start, impute=impute)

    fold_metrics = []
    all_y_true, all_y_pred = [], []

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param('features',         feature_cols)
        mlflow.log_param('n_features',       len(feature_cols))
        mlflow.log_param('model_type',       type(model).__name__)
        mlflow.log_param('val_start_season', val_start)
        mlflow.log_param('n_folds',          len(folds))
        mlflow.log_param('gender',           GENDER)
        mlflow.log_param('scale',            scale)

        if hasattr(model, 'get_params'):
            mlflow.log_params({
                f'model__{k}': v for k, v in model.get_params().items()
            })
        if tags:
            mlflow.set_tags(tags)

        for val_season, X_train, y_train, X_val, y_val, sample_weights in folds:
            fit_params = {}
            if sample_weights is not None:
                fit_params['model__sample_weight'] = sample_weights

            pipeline.fit(X_train, y_train, **fit_params)
            y_pred = pipeline.predict_proba(X_val)[:, 1]
            
            metrics = compute_metrics(y_val, y_pred)
            metrics['season'] = val_season
            fold_metrics.append(metrics)
            all_y_true.append(y_val.values)
            all_y_pred.append(y_pred)

            mlflow.log_metrics({
                f'fold_{val_season}_brier':    metrics['brier_score'],
                f'fold_{val_season}_logloss':  metrics['log_loss'],
                f'fold_{val_season}_auc':      metrics['roc_auc'],
                f'fold_{val_season}_accuracy': metrics['accuracy'],
            })

        metrics_df = pd.DataFrame(fold_metrics)
        agg = {
            'mean_brier':    metrics_df['brier_score'].mean(),
            'std_brier':     metrics_df['brier_score'].std(),
            'min_brier':     metrics_df['brier_score'].min(),
            'max_brier':     metrics_df['brier_score'].max(),
            'mean_logloss':  metrics_df['log_loss'].mean(),
            'mean_auc':      metrics_df['roc_auc'].mean(),
            'mean_accuracy': metrics_df['accuracy'].mean(),
        }
        modern_folds = metrics_df[metrics_df['season'] >= 2021]
        agg['modern_mean_brier'] = modern_folds['brier_score'].mean()
        agg['modern_std_brier']  = modern_folds['brier_score'].std()
        agg['modern_mean_auc']   = modern_folds['roc_auc'].mean()
        mlflow.log_metrics(agg)

        fig = plot_calibration(all_y_true, all_y_pred, run_name)
        mlflow.log_figure(fig, 'calibration_curve.png')
        plt.close(fig)

        final_train_df = augment_matchups(matchups_df.copy())
        
        if impute:
            X_all = final_train_df[feature_cols].fillna(final_train_df[feature_cols].median())
        else:
            X_all = final_train_df[feature_cols]
            
        y_all = final_train_df['Label']
        
        final_fit_params = {}
        
        if season_decay is not None:
            max_season = final_train_df['Season'].max()
            
            fresh_weights = season_decay ** (max_season - final_train_df['Season'])
            fresh_weights = fresh_weights / fresh_weights.mean()
            
            final_fit_params['model__sample_weight'] = fresh_weights

        pipeline.fit(X_all, y_all, **final_fit_params)
        mlflow.sklearn.log_model(pipeline, 'model')

    print(f'[{run_name}]')
    print(f'  Mean Brier (2016–2025) : {agg["mean_brier"]:.4f} ± {agg["std_brier"]:.4f}')
    print(f'  Mean Brier (2021–2025) : {agg["modern_mean_brier"]:.4f} ± {agg["modern_std_brier"]:.4f}')
    print(f'  Mean LogLoss           : {agg["mean_logloss"]:.4f}')
    print(f'  Mean AUC               : {agg["mean_auc"]:.4f}')
    print(f'  Mean Acc               : {agg["mean_accuracy"]:.4f}')
    print()

    return metrics_df, pipeline

## Model, Feature Sweep

In [21]:
import lightgbm as lgb

lgbm_baseline = lgb.LGBMClassifier(
    n_estimators=100, 
    learning_rate=0.05, 
    max_depth=4,         # Keep it shallow for the baseline
    num_leaves=15, 
    random_state=42,
    verbose=-1           # Shuts off the LightGBM warning spam
)

# 1. The Core (The undisputed champion from your last sweep)
core_features = [
    'SeedA', 
    'SeedB', 
    'SeedDiff', 
    'EloA', 
    'EloB', 
    'EloDiff', 
    'MasseyMedianDiff', 
    'MasseyMeanDiff', 
    'MasseyMinDiff', 
    'MasseyMaxDiff'
]

# 2. Season Form & Efficiency
season_form = [
    'Season_WinRate_Diff', 'Season_AvgMargin_Diff', 'Season_AvgOffEff_Diff', 
    'Season_AvgDefEff_Diff', 'Season_EffMargin_Diff', 'Season_AvgPace_Diff'
]

# 3. Momentum & Trend (How hot are they right now?)
last_10_trend = [
    'Last10_WinRate_Diff', 'Last10_AvgMargin_Diff', 'Last10_AvgOffEff_Diff', 
    'Last10_AvgDefEff_Diff', 'Last10_EffMargin_Diff', 'Last10_AvgPace_Diff',
    'Trend_EffMargin_Diff', 'Trend_WinRate_Diff'
]

# 4. Strength of Schedule (Who have they actually played?)
sos_features = [
    'SOS_Season_AvgOppElo_Diff', 'SOS_Season_AvgOppEffMargin_Diff', 'SOS_Season_AvgOppWinRate_Diff',
    'SOS_Last10_AvgOppElo_Diff', 'SOS_Last10_AvgOppEffMargin_Diff', 'SOS_Last10_AvgOppWinRate_Diff'
]

# 5. Coaching Experience
coach_features = [
    'Coach_TourneyApps_Diff', 'Coach_TourneyWins_Diff', 
    'Coach_TourneyWinRate_Diff', 'Coach_TourneyGames_Diff'
]

# 6. Four Factors & Shooting (The granular basketball metrics)
four_factors = [
    'Season_EFG_Diff', 'Season_EFGD_Diff', 'Season_TOR_Diff', 'Season_TORD_Diff', 
    'Season_ORB_Diff', 'Season_DRB_Diff', 'Season_FTR_Diff', 'Season_FTRD_Diff', 
    'Season_3PR_Diff', 'Season_3PRD_Diff', 'Season_3P_Diff', 'Season_3PD_Diff'
]

# --- Build the Test Dictionary ---
forward_selection_tiers = {
    "0_Baseline_Core": core_features,
    "1_Core_+_SeasonForm": core_features + season_form,
    "2_Core_+_Last10_Trend": core_features + last_10_trend,
    "3_Core_+_SOS": core_features + sos_features,
    "4_Core_+_Coaches": core_features + coach_features,
    "5_Core_+_FourFactors": core_features + four_factors,
    "6_The_Everything_Bagel": core_features + season_form + last_10_trend + sos_features + coach_features + four_factors
}

# --- Run the Sweep ---
print("🚀 Starting Forward Feature Selection on LightGBM...\n")
forward_results = []

for tier_name, features in forward_selection_tiers.items():
    print(f"🔄 Evaluating: {tier_name} ({len(features)} features)")
    
    metrics_df, _ = run_experiment(
        run_name=f"LGBM_{tier_name}",
        model=lgbm_baseline,         # Your LightGBM model
        feature_cols=features,
        matchups_df=matchups,        # Using ALL historical data
        val_start=2016,
        scale=False,                 # No scale needed for LGBM
        impute=False                 # Let LGBM handle NaNs!
    )
    
    # Track the Modern Era Brier Score (2021-2025)
    modern_brier = metrics_df[metrics_df['season'] >= 2021]['brier_score'].mean()
    forward_results.append({'Run': tier_name, 'Modern_Brier': modern_brier, 'Num_Features': len(features)})

print("\n🏆 --- FORWARD SELECTION COMPLETE --- 🏆")
results_df = pd.DataFrame(forward_results).sort_values('Modern_Brier')
print(results_df.to_string(index=False))

🚀 Starting Forward Feature Selection on LightGBM...

🔄 Evaluating: 0_Baseline_Core (10 features)
[LGBM_0_Baseline_Core]
  Mean Brier (2016–2025) : 0.1869 ± 0.0247
  Mean Brier (2021–2025) : 0.1953 ± 0.0283
  Mean LogLoss           : 0.5559
  Mean AUC               : 0.7895
  Mean Acc               : 0.7125

🔄 Evaluating: 1_Core_+_SeasonForm (16 features)
[LGBM_1_Core_+_SeasonForm]
  Mean Brier (2016–2025) : 0.1885 ± 0.0279
  Mean Brier (2021–2025) : 0.1988 ± 0.0318
  Mean LogLoss           : 0.5584
  Mean AUC               : 0.7863
  Mean Acc               : 0.7142

🔄 Evaluating: 2_Core_+_Last10_Trend (18 features)
[LGBM_2_Core_+_Last10_Trend]
  Mean Brier (2016–2025) : 0.1905 ± 0.0251
  Mean Brier (2021–2025) : 0.1994 ± 0.0285
  Mean LogLoss           : 0.5671
  Mean AUC               : 0.7817
  Mean Acc               : 0.7109

🔄 Evaluating: 3_Core_+_SOS (16 features)
[LGBM_3_Core_+_SOS]
  Mean Brier (2016–2025) : 0.1892 ± 0.0297
  Mean Brier (2021–2025) : 0.1976 ± 0.0360
  Mean LogLo

In [22]:
import random
import lightgbm as lgb
import pandas as pd

# 1. Define the LightGBM Search Space
# We are focusing on regularization to prevent overfitting on the larger feature sets
param_dist = {
    'learning_rate': [0.01, 0.03, 0.05, 0.08],
    'max_depth': [3, 4, 5, 6],
    'num_leaves': [7, 15, 31, 63],
    'min_child_samples': [10, 30, 50, 100],  # Higher = more conservative
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0], # Feature fraction
    'subsample': [0.7, 0.8, 0.9, 1.0]         # Bagging fraction
}

n_iter = 10 # Number of random hyperparameter combos to test per feature set
tuned_results = []

print(f"🚀 Starting LightGBM Hyperparameter Sweep ({n_iter} iterations per set)...\n")

for tier_name, features in forward_selection_tiers.items():
    print(f"🔍 Tuning: {tier_name} ({len(features)} features)")
    
    best_modern_brier = float('inf')
    best_all_brier = float('inf')
    best_params = {}
    
    for i in range(n_iter):
        # Sample random hyperparameters
        params = {
            'n_estimators': 150, # Give it enough trees to learn with lower learning rates
            'learning_rate': random.choice(param_dist['learning_rate']),
            'max_depth': random.choice(param_dist['max_depth']),
            'min_child_samples': random.choice(param_dist['min_child_samples']),
            'colsample_bytree': random.choice(param_dist['colsample_bytree']),
            'subsample': random.choice(param_dist['subsample']),
            'random_state': 42,
            'verbose': -1
        }
        
        # Prevent num_leaves from exceeding the max possible for the chosen depth
        max_possible_leaves = 2 ** params['max_depth'] - 1
        params['num_leaves'] = min(random.choice(param_dist['num_leaves']), max_possible_leaves)
        
        # Build the model
        model = lgb.LGBMClassifier(**params)
        
        # Run your pipeline
        # (Silencing MLflow run_name slightly so it doesn't clutter your dashboard completely, 
        # or you can leave it to log all 70 runs!)
        metrics_df, _ = run_experiment(
            run_name=f"Tune_{tier_name}_{i}",
            model=model,
            feature_cols=features,
            matchups_df=matchups,
            val_start=2016,
            scale=False,
            impute=False
        )
        
        # Extract metrics
        modern_brier = metrics_df[metrics_df['season'] >= 2021]['brier_score'].mean()
        all_brier = metrics_df['brier_score'].mean()
        
        # Keep track of the best model based on Modern Era performance
        if modern_brier < best_modern_brier:
            best_modern_brier = modern_brier
            best_all_brier = all_brier
            best_params = params.copy()
            
    print(f"   ✅ Best Modern: {best_modern_brier:.5f} | All Brier: {best_all_brier:.5f}")
    
    # Format params for clean display
    param_str = f"LR:{best_params['learning_rate']}, Depth:{best_params['max_depth']}, FeatFrac:{best_params['colsample_bytree']}"
    
    tuned_results.append({
        'Run': tier_name,
        'Modern_Brier': best_modern_brier,
        'Mean_All_Brier': best_all_brier,
        'Num_Features': len(features),
        'Best_Key_Params': param_str
    })

print("\n🏆 --- TUNING SWEEP COMPLETE --- 🏆")
results_df = pd.DataFrame(tuned_results).sort_values('Modern_Brier')
print(results_df.to_string(index=False))

🚀 Starting LightGBM Hyperparameter Sweep (10 iterations per set)...

🔍 Tuning: 0_Baseline_Core (10 features)
[Tune_0_Baseline_Core_0]
  Mean Brier (2016–2025) : 0.1875 ± 0.0253
  Mean Brier (2021–2025) : 0.1965 ± 0.0298
  Mean LogLoss           : 0.5581
  Mean AUC               : 0.7883
  Mean Acc               : 0.7109

[Tune_0_Baseline_Core_1]
  Mean Brier (2016–2025) : 0.1872 ± 0.0267
  Mean Brier (2021–2025) : 0.1943 ± 0.0325
  Mean LogLoss           : 0.5565
  Mean AUC               : 0.7906
  Mean Acc               : 0.7092

[Tune_0_Baseline_Core_2]
  Mean Brier (2016–2025) : 0.1928 ± 0.0199
  Mean Brier (2021–2025) : 0.1992 ± 0.0237
  Mean LogLoss           : 0.5692
  Mean AUC               : 0.7846
  Mean Acc               : 0.7126

[Tune_0_Baseline_Core_3]
  Mean Brier (2016–2025) : 0.1885 ± 0.0260
  Mean Brier (2021–2025) : 0.1964 ± 0.0311
  Mean LogLoss           : 0.5581
  Mean AUC               : 0.7890
  Mean Acc               : 0.7192

[Tune_0_Baseline_Core_4]
  Mean Bri

In [26]:
import itertools
import random
import lightgbm as lgb
import xgboost as xgb
import pandas as pd
from sklearn.metrics import brier_score_loss

# 1. Define the Feature Sets
base_features = [
    'SeedA', 'SeedB', 'SeedDiff', 
    'EloA', 'EloB', 'EloDiff', 
    'MasseyMedianDiff', 'MasseyMeanDiff', 'MasseyMinDiff', 'MasseyMaxDiff'
]

sos_features = [
    'SOS_Season_AvgOppElo_Diff', 'SOS_Season_AvgOppEffMargin_Diff', 'SOS_Season_AvgOppWinRate_Diff',
    'SOS_Last10_AvgOppElo_Diff', 'SOS_Last10_AvgOppEffMargin_Diff', 'SOS_Last10_AvgOppWinRate_Diff'
]

season_form = [
    'Season_WinRate_Diff', 'Season_AvgMargin_Diff', 'Season_AvgOffEff_Diff', 
    'Season_AvgDefEff_Diff', 'Season_EffMargin_Diff', 'Season_AvgPace_Diff'
]

last_10_trend = [
    'Last10_WinRate_Diff', 'Last10_AvgMargin_Diff', 'Last10_AvgOffEff_Diff', 
    'Last10_AvgDefEff_Diff', 'Last10_EffMargin_Diff', 'Last10_AvgPace_Diff',
    'Trend_EffMargin_Diff', 'Trend_WinRate_Diff'
]

coach_features = [
    'Coach_TourneyApps_Diff', 'Coach_TourneyWins_Diff', 
    'Coach_TourneyWinRate_Diff', 'Coach_TourneyGames_Diff'
]

four_factors = [
    'Season_EFG_Diff', 'Season_EFGD_Diff', 'Season_TOR_Diff', 'Season_TORD_Diff', 
    'Season_ORB_Diff', 'Season_DRB_Diff', 'Season_FTR_Diff', 'Season_FTRD_Diff', 
    'Season_3PR_Diff', 'Season_3PRD_Diff', 'Season_3P_Diff', 'Season_3PD_Diff'
]

optional_groups = {
    "SOS": sos_features,
    "Form": season_form,
    "Trend": last_10_trend,
    "Coaches": coach_features,
    "FourFactors": four_factors
}

# 2. Build the 32 Combinations
all_combos = {"Baseline_Core_Only": base_features} 
group_names = list(optional_groups.keys())
for r in range(1, len(group_names) + 1):
    for combo_names in itertools.combinations(group_names, r):
        combo_name = "Core_+_" + "_+_".join(combo_names)
        combo_features = list(base_features) 
        for name in combo_names:
            combo_features.extend(optional_groups[name])
        all_combos[combo_name] = combo_features

# 3. Define Parameter Distributions
lgbm_param_dist = {
    'learning_rate': [0.01, 0.03, 0.05, 0.08],
    'max_depth': [3, 4, 5, 6],
    'num_leaves': [7, 15, 31, 63],
    'min_child_samples': [10, 30, 50, 100],  
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0], 
    'subsample': [0.7, 0.8, 0.9, 1.0]         
}

xgb_param_dist = {
    'learning_rate': [0.01, 0.03, 0.05, 0.08],
    'max_depth': [3, 4, 5, 6],
    'min_child_weight': [1, 3, 5, 7],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0], 
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

# 4. Split Data into CV (tuning) and Holdout (testing)
cv_matchups = matchups[matchups['Season'] <= 2023].copy()
holdout_matchups = matchups[matchups['Season'] >= 2024].copy()

# 5. Execute the Dual-Sweep
exhaustive_results = []
models_to_test = ['LightGBM', 'XGBoost']

print("Starting exhaustive sweep with strict holdout verification...\n")

for combo_name, features in all_combos.items():
    print(f"Testing: {combo_name} ({len(features)} features)")
    
    for model_type in models_to_test:
        best_cv_brier = float('inf')
        best_params = {}
        
        # 5a. Hyperparameter Search Loop
        for i in range(10): 
            if model_type == 'LightGBM':
                params = {
                    'n_estimators': 150,
                    'learning_rate': random.choice(lgbm_param_dist['learning_rate']),
                    'max_depth': random.choice(lgbm_param_dist['max_depth']),
                    'min_child_samples': random.choice(lgbm_param_dist['min_child_samples']),
                    'colsample_bytree': random.choice(lgbm_param_dist['colsample_bytree']),
                    'subsample': random.choice(lgbm_param_dist['subsample']),
                    'random_state': 42 + i,
                    'verbose': -1
                }
                max_possible_leaves = 2 ** params['max_depth'] - 1
                params['num_leaves'] = min(random.choice(lgbm_param_dist['num_leaves']), max_possible_leaves)
                model = lgb.LGBMClassifier(**params)
                
            else: # XGBoost
                params = {
                    'n_estimators': 150,
                    'learning_rate': random.choice(xgb_param_dist['learning_rate']),
                    'max_depth': random.choice(xgb_param_dist['max_depth']),
                    'min_child_weight': random.choice(xgb_param_dist['min_child_weight']),
                    'colsample_bytree': random.choice(xgb_param_dist['colsample_bytree']),
                    'subsample': random.choice(xgb_param_dist['subsample']),
                    'random_state': 42 + i,
                    'eval_metric': 'logloss'
                }
                model = xgb.XGBClassifier(**params)
            
            # Run Walk-Forward CV on the restricted dataset (up to 2023)
            short_params = f"LR{params['learning_rate']}_D{params['max_depth']}_F{params['colsample_bytree']}"

            # 2. Combine it with the model type (and optionally the combo_name)
            current_run_name = f"{model_type}_{short_params}"
            metrics_df, _ = run_experiment(
                run_name=current_run_name, 
                model=model,
                feature_cols=features,
                matchups_df=cv_matchups,
                val_start=2016,
                scale=False,
                impute=False
            )
            
            cv_brier = metrics_df['brier_score'].mean()
            
            if cv_brier < best_cv_brier:
                best_cv_brier = cv_brier
                best_params = params.copy()
                
        # 5b. Holdout Verification
        # Retrain a model with the best parameters on the full augmented CV set
        if model_type == 'LightGBM':
            final_cv_model = lgb.LGBMClassifier(**best_params)
        else:
            final_cv_model = xgb.XGBClassifier(**best_params)
            
        augmented_cv = augment_matchups(cv_matchups.copy())
        X_train_cv = augmented_cv[features]
        y_train_cv = augmented_cv['Label']
        final_cv_model.fit(X_train_cv, y_train_cv)
        
        # Predict on the untouched 2024-2025 holdout set
        X_holdout = holdout_matchups[features]
        y_holdout = holdout_matchups['Label']
        preds_holdout = final_cv_model.predict_proba(X_holdout)[:, 1]
        holdout_brier = brier_score_loss(y_holdout, preds_holdout)
        
        param_str = f"LR:{best_params['learning_rate']}, Depth:{best_params['max_depth']}, FeatFrac:{best_params['colsample_bytree']}"
        
        exhaustive_results.append({
            'Model': model_type,
            'Combo': combo_name,
            'CV_Brier': best_cv_brier,
            'Holdout_Brier': holdout_brier,
            'Num_Features': len(features),
            'Best_Params': param_str
        })

# 6. Output the Results
print("\n--- EXHAUSTIVE SWEEP COMPLETE ---")
results_df = pd.DataFrame(exhaustive_results)
# Sort by Holdout Brier to identify the models that generalize best
results_df = results_df.sort_values('Holdout_Brier')
print(results_df.head(15).to_string(index=False))

Starting exhaustive sweep with strict holdout verification...

Testing: Baseline_Core_Only (10 features)
[LightGBM_LR0.05_D5_F0.8]
  Mean Brier (2016–2025) : 0.1941 ± 0.0244
  Mean Brier (2021–2025) : 0.2165 ± 0.0075
  Mean LogLoss           : 0.5770
  Mean AUC               : 0.7776
  Mean Acc               : 0.7072

[LightGBM_LR0.08_D3_F1.0]
  Mean Brier (2016–2025) : 0.1949 ± 0.0225
  Mean Brier (2021–2025) : 0.2152 ± 0.0083
  Mean LogLoss           : 0.5799
  Mean AUC               : 0.7752
  Mean Acc               : 0.7049

[LightGBM_LR0.03_D4_F0.6]
  Mean Brier (2016–2025) : 0.1931 ± 0.0221
  Mean Brier (2021–2025) : 0.2129 ± 0.0053
  Mean LogLoss           : 0.5713
  Mean AUC               : 0.7841
  Mean Acc               : 0.7072

[LightGBM_LR0.05_D4_F0.7]
  Mean Brier (2016–2025) : 0.1950 ± 0.0237
  Mean Brier (2021–2025) : 0.2159 ± 0.0094
  Mean LogLoss           : 0.5827
  Mean AUC               : 0.7760
  Mean Acc               : 0.6964

[LightGBM_LR0.01_D4_F1.0]
  Mean Br

## Ready for Submission

In [81]:
print("Initializing Final 2026 Men's Ensemble...")

# 1. Define the Winning Feature Sets from your Sweep
features_xgb_heavy = base_features + sos_features + season_form + coach_features 
features_lgbm_robust = base_features + sos_features
features_anchor = base_features

full_train_df = augment_matchups(matchups.copy())

print("Training Model 1")
model_xgb = xgb.XGBClassifier(
    n_estimators=150, learning_rate=0.05, max_depth=4, 
    colsample_bytree=0.8, subsample=0.8, random_state=42
)
model_xgb.fit(full_train_df[features_xgb_heavy], full_train_df['Label'])

print("Training Model 2")
model_lgbm = lgb.LGBMClassifier(
    n_estimators=150, learning_rate=0.05, max_depth=3, 
    colsample_bytree=0.6, subsample=0.7, random_state=42, verbose=-1
)
model_lgbm.fit(full_train_df[features_lgbm_robust], full_train_df['Label'])

print("Training Model 3")
model_anchor = lgb.LGBMClassifier(
    n_estimators=150, learning_rate=0.03, max_depth=4, 
    colsample_bytree=0.6, random_state=42, verbose=-1
)
model_anchor.fit(full_train_df[features_anchor], full_train_df['Label'])

print("Processing 2026 Matchups...")
sub_df = pd.read_csv("../data/raw/SampleSubmissionStage2.csv")
sub_df['Season'] = sub_df['ID'].apply(lambda x: int(x.split('_')[0]))
sub_df['WTeamID'] = sub_df['ID'].apply(lambda x: int(x.split('_')[1]))
sub_df['LTeamID'] = sub_df['ID'].apply(lambda x: int(x.split('_')[2]))
mens_sub = sub_df[sub_df['WTeamID'] < 2000].copy()

# Create the 2026 Submission features
sub_features_df = build_matchup_features(mens_sub, elo_ratings, seed_lookup, form_df, consensus_df, coach_df, sos_df, ff_df)

print("🔮 Generating Ensemble Predictions...")
pred_xgb = model_xgb.predict_proba(sub_features_df[features_xgb_heavy])[:, 1]
pred_lgbm = model_lgbm.predict_proba(sub_features_df[features_lgbm_robust])[:, 1]
pred_anchor = model_anchor.predict_proba(sub_features_df[features_anchor])[:, 1]

mens_sub['Pred'] = (pred_xgb * 0.4) + (pred_lgbm * 0.4) + (pred_anchor * 0.2)

final_sub = mens_sub[['ID', 'Pred']].copy()
final_sub.to_csv("submission_mens_2026.csv", index=False)

print(f"\n🎉 SUCCESS! Generated {len(final_sub)} predictions.")
print("Top 5 Matchups Preview:")
print(final_sub.head())

Initializing Final 2026 Men's Ensemble...
Training Model 1
Training Model 2
Training Model 3
Processing 2026 Matchups...
🔮 Generating Ensemble Predictions...

🎉 SUCCESS! Generated 66430 predictions.
Top 5 Matchups Preview:
               ID      Pred
0  2026_1101_1102  0.720910
1  2026_1101_1103  0.088047
2  2026_1101_1104  0.073403
3  2026_1101_1105  0.699694
4  2026_1101_1106  0.786640


## Simulation

In [ ]:
print("📂 Loading Tournament Data...")
slots = pd.read_csv("../data/raw/MNCAATourneySlots.csv")
seeds = pd.read_csv("../data/raw/MNCAATourneySeeds.csv")
teams = pd.read_csv("../data/raw/MTeams.csv")

# Create lookups for speed
# final_sub is your ensemble results dataframe
prob_lookup = final_sub.set_index('ID')['Pred'].to_dict()
team_names = teams.set_index('TeamID')['TeamName'].to_dict()
seeds_2026_base = seeds[seeds['Season'] == 2026].set_index('Seed')['TeamID'].to_dict()
slots_2026 = slots[slots['Season'] == 2026].copy()

# 2. THE CORE SIMULATOR FUNCTION
def simulate_tournament(slots_df, seeds_dict, p_lookup):
    """
    Simulates one full tournament bracket using high-speed dict lookups.
    """
    # Copy seeds so we don't mutate the original across simulations
    curr_seeds = seeds_dict.copy()
    results = {} 

    # --- PHASE 1: Resolve First Four (Play-In Games) ---
    # These are slots like 'X16' that depend on 'X16a' and 'X16b'
    play_in_slots = ['X16', 'Y11', 'Y16', 'Z11']
    for slot_name in play_in_slots:
        slot_info = slots_df[slots_df['Slot'] == slot_name].iloc[0]
        t_a, t_b = curr_seeds[slot_info['StrongSeed']], curr_seeds[slot_info['WeakSeed']]
        
        low, high = min(t_a, t_b), max(t_a, t_b)
        prob_a = p_lookup[f"2026_{low}_{high}"]
        if t_a != low: prob_a = 1 - prob_a
        
        curr_seeds[slot_name] = t_a if np.random.random() < prob_a else t_b

    # --- PHASE 2: Round 1 (64 Teams) ---
    r1_slots = slots_df[slots_df['Slot'].str.startswith('R1')]
    for _, slot in r1_slots.iterrows():
        t_a, t_b = curr_seeds[slot['StrongSeed']], curr_seeds[slot['WeakSeed']]
        low, high = min(t_a, t_b), max(t_a, t_b)
        prob_a = p_lookup[f"2026_{low}_{high}"]
        if t_a != low: prob_a = 1 - prob_a
        
        results[slot['Slot']] = t_a if np.random.random() < prob_a else t_b

    # --- PHASE 3: Rounds 2-6 (The Gauntlet) ---
    for r in range(2, 7):
        round_slots = slots_df[slots_df['Slot'].str.startswith(f'R{r}')]
        for _, slot in round_slots.iterrows():
            t_a, t_b = results[slot['StrongSeed']], results[slot['WeakSeed']]
            low, high = min(t_a, t_b), max(t_a, t_b)
            prob_a = p_lookup[f"2026_{low}_{high}"]
            if t_a != low: prob_a = 1 - prob_a
            
            results[slot['Slot']] = t_a if np.random.random() < prob_a else t_b
            
    # Return as a tuple of (Slot, WinnerID) so it can be hashed/counted
    return tuple(sorted(results.items()))

# 3. EXECUTION: RUN 10,000 SIMULATIONS
print(f"🎲 Simulating 10,000 brackets for 2026...")
all_brackets = []

# Using a standard loop (or replace with range(10000) if no tqdm)
for i in range(10000):
    bracket = simulate_tournament(slots_2026, seeds_2026_base, prob_lookup)
    all_brackets.append(bracket)
    if (i+1) % 1000 == 0:
        print(f"  > {i+1} brackets simulated...")

# 4. ANALYZE THE RESULTS
bracket_counts = Counter(all_brackets)
top_5 = bracket_counts.most_common(5)

print("\n🏆 --- SIMULATION COMPLETE: TOP 5 MOST PROBABLE BRACKETS ---")
for i, (bracket, count) in enumerate(top_5):
    winner_id = [v for k, v in bracket if k == 'R6CH'][0]
    winner_name = team_names[winner_id]
    print(f"Rank {i+1}: Winner = {winner_name} | Freq = {count}/10000")

📂 Loading Tournament Data...
🎲 Simulating 10,000 brackets for 2026...
  > 2500 brackets simulated...
  > 5000 brackets simulated...
  > 7500 brackets simulated...
  > 10000 brackets simulated...
  > 12500 brackets simulated...
  > 15000 brackets simulated...
  > 17500 brackets simulated...
  > 20000 brackets simulated...
  > 22500 brackets simulated...
  > 25000 brackets simulated...
  > 27500 brackets simulated...
  > 30000 brackets simulated...
  > 32500 brackets simulated...
  > 35000 brackets simulated...
  > 37500 brackets simulated...
  > 40000 brackets simulated...
  > 42500 brackets simulated...
  > 45000 brackets simulated...
  > 47500 brackets simulated...
  > 50000 brackets simulated...
  > 52500 brackets simulated...
  > 55000 brackets simulated...
  > 57500 brackets simulated...
  > 60000 brackets simulated...
  > 62500 brackets simulated...
  > 65000 brackets simulated...
  > 67500 brackets simulated...
  > 70000 brackets simulated...
  > 72500 brackets simulated...
  > 7

In [79]:
# 1. Track how often each team reaches each round
from collections import defaultdict

# round_reach[team_id][round_name] = count
round_reach = defaultdict(lambda: defaultdict(int))

for bracket in all_brackets:
    for slot, winner in bracket:
        # Check which round the slot belongs to
        round_prefix = slot[:2] # 'R1', 'R2', etc.
        round_reach[winner][round_prefix] += 1

# 2. Print the most frequent Champions (R6)
print("🏆 --- MOST FREQUENT NATIONAL CHAMPIONS ---")
champ_counts = []
for team_id, rounds in round_reach.items():
    if 'R6' in rounds:
        champ_counts.append((team_names[team_id], rounds['R6']))

# Sort by count descending
champ_counts.sort(key=lambda x: x[1], reverse=True)

for name, count in champ_counts[:10]:
    print(f"{name}: {count} wins ({count/100000:.1%})")

# 3. Print the most frequent Final Four teams (R4 winners)
print("\n🔥 --- MOST FREQUENT FINAL FOUR TEAMS ---")
f4_counts = []
for team_id, rounds in round_reach.items():
    if 'R4' in rounds:
        f4_counts.append((team_names[team_id], rounds['R4']))

f4_counts.sort(key=lambda x: x[1], reverse=True)

for name, count in f4_counts[:10]:
    print(f"{name}: {count} appearances ({count/100000:.1%})")

🏆 --- MOST FREQUENT NATIONAL CHAMPIONS ---
Michigan: 30010 wins (30.0%)
Duke: 20284 wins (20.3%)
Arizona: 17210 wins (17.2%)
Houston: 5789 wins (5.8%)
Florida: 5754 wins (5.8%)
Purdue: 3108 wins (3.1%)
Iowa St: 2839 wins (2.8%)
Gonzaga: 2608 wins (2.6%)
Connecticut: 1832 wins (1.8%)
Illinois: 1710 wins (1.7%)

🔥 --- MOST FREQUENT FINAL FOUR TEAMS ---
Michigan: 63681 appearances (63.7%)
Duke: 55883 appearances (55.9%)
Arizona: 50592 appearances (50.6%)
Florida: 28484 appearances (28.5%)
Houston: 26518 appearances (26.5%)
Purdue: 18016 appearances (18.0%)
Gonzaga: 17168 appearances (17.2%)
Iowa St: 13631 appearances (13.6%)
Connecticut: 13467 appearances (13.5%)
Illinois: 13192 appearances (13.2%)


In [80]:
from collections import defaultdict

# 1. Initialize a counter for each team's total wins across all simulations
total_wins_per_team = defaultdict(int)

print("📊 Calculating Expected Wins (EV) from 10,000 simulations...")

for bracket in all_brackets:
    # Each 'bracket' is a tuple of (Slot, WinnerID)
    # We count how many times each WinnerID appears in that single bracket
    for slot, winner_id in bracket:
        total_wins_per_team[winner_id] += 1

# 2. Convert to Average (EV)
num_sims = len(all_brackets)
ev_results = []

for team_id, total_wins in total_wins_per_team.items():
    ev = total_wins / num_sims
    ev_results.append({
        'Team': team_names[team_id],
        'ExpectedWins': ev
    })

# 3. Display the Top 15 Teams by EV
ev_df = pd.DataFrame(ev_results).sort_values(by='ExpectedWins', ascending=False)

print("\n🏆 --- TOP TEAMS BY EXPECTED WINS (EV) ---")
print(ev_df.head(64).to_string(index=False))

📊 Calculating Expected Wins (EV) from 10,000 simulations...

🏆 --- TOP TEAMS BY EXPECTED WINS (EV) ---
          Team  ExpectedWins
      Michigan       4.00495
          Duke       3.71058
       Arizona       3.45163
       Florida       2.64516
       Houston       2.50678
       Gonzaga       2.41871
        Purdue       2.40269
       Iowa St       2.24276
   Connecticut       2.23547
      Illinois       1.97370
   Michigan St       1.95824
      Virginia       1.88250
    Vanderbilt       1.71980
     St John's       1.62543
      Arkansas       1.60987
       Alabama       1.44143
        Kansas       1.38722
      Nebraska       1.33682
     Wisconsin       1.23820
    Texas Tech       1.21899
  St Mary's CA       1.15444
     Tennessee       1.12565
          UCLA       1.01721
    Louisville       0.94603
      Miami FL       0.88827
North Carolina       0.85259
      Kentucky       0.82785
           BYU       0.82144
       Utah St       0.80408
          Iowa       0.8026

In [75]:
import pandas as pd
import numpy as np

def generate_and_print_bracket(sub_df, slots_path, seeds_path, teams_path):
    # 1. Load Data
    slots = pd.read_csv(slots_path)
    seeds = pd.read_csv(seeds_path)
    teams = pd.read_csv(teams_path)
    
    # 2. Setup Lookups
    # Filters for 2026 and creates fast-access dictionaries
    prob_lookup = sub_df.set_index('ID')['Pred'].to_dict()
    team_names = teams.set_index('TeamID')['TeamName'].to_dict()
    seeds_2026 = seeds[seeds['Season'] == 2026].set_index('Seed')['TeamID'].to_dict()
    slots_2026 = slots[slots['Season'] == 2026].copy()
    
    bracket_results = {} # To store Slot -> WinnerID

    # 3. Step 0: Resolve First Four (Play-Ins)
    # These are slots in your CSV that act as Seeds for Round 1
    play_in_slots = ['X16', 'Y11', 'Y16', 'Z11']
    print("--- FIRST FOUR (PLAY-INS) ---")
    for slot_name in play_in_slots:
        slot_info = slots_2026[slots_2026['Slot'] == slot_name].iloc[0]
        t_a = seeds_2026[slot_info['StrongSeed']]
        t_b = seeds_2026[slot_info['WeakSeed']]
        
        low, high = min(t_a, t_b), max(t_a, t_b)
        prob_a = prob_lookup[f"2026_{low}_{high}"]
        if t_a != low: prob_a = 1 - prob_a
        
        # We pick the 50%+ favorite for the Chalk Bracket
        winner = t_a if prob_a >= 0.5 else t_b
        seeds_2026[slot_name] = winner # Replace seed with the actual team
        print(f"Slot {slot_name}: {team_names[t_a]} vs {team_names[t_b]} -> {team_names[winner]} ({prob_a:.1%})")

    # 4. Step 1-6: Round by Round Tournament
    for r in range(1, 7):
        print(f"\n--- ROUND {r} ---")
        round_slots = slots_2026[slots_2026['Slot'].str.startswith(f'R{r}')]
        
        for _, slot in round_slots.iterrows():
            slot_id = slot['Slot']
            
            # Determine participants
            if r == 1:
                t_a = seeds_2026[slot['StrongSeed']]
                t_b = seeds_2026[slot['WeakSeed']]
            else:
                t_a = bracket_results[slot['StrongSeed']]
                t_b = bracket_results[slot['WeakSeed']]
            
            # Fetch Ensemble Probability
            low, high = min(t_a, t_b), max(t_a, t_b)
            prob_low_wins = prob_lookup[f"2026_{low}_{high}"]
            prob_a = prob_low_wins if t_a == low else (1 - prob_low_wins)
            
            # Record Winner
            winner = t_a if prob_a >= 0.5 else t_b
            bracket_results[slot_id] = winner
            
            print(f"{slot_id}: {team_names[t_a]} vs {team_names[t_b]} -> **{team_names[winner]}** ({prob_a:.1%})")

# --- EXECUTION ---
# Ensure the file paths match your local directory structure
generate_and_print_bracket(
    final_sub, 
    "../data/raw/MNCAATourneySlots.csv", 
    "../data/raw/MNCAATourneySeeds.csv", 
    "../data/raw/MTeams.csv"
)

--- FIRST FOUR (PLAY-INS) ---
Slot X16: Lehigh vs Prairie View -> Lehigh (55.9%)
Slot Y11: Miami OH vs SMU -> Miami OH (56.7%)
Slot Y16: Howard vs UMBC -> UMBC (47.9%)
Slot Z11: NC State vs Texas -> NC State (52.3%)

--- ROUND 1 ---
R1W1: Duke vs Siena -> **Duke** (96.0%)
R1W2: Connecticut vs Furman -> **Connecticut** (95.9%)
R1W3: Michigan St vs N Dakota St -> **Michigan St** (87.2%)
R1W4: Kansas vs Cal Baptist -> **Kansas** (86.3%)
R1W5: St John's vs Northern Iowa -> **St John's** (78.3%)
R1W6: Louisville vs South Florida -> **Louisville** (57.6%)
R1W7: UCLA vs UCF -> **UCLA** (66.8%)
R1W8: Ohio St vs TCU -> **Ohio St** (52.5%)
R1X1: Florida vs Lehigh -> **Florida** (97.2%)
R1X2: Houston vs Idaho -> **Houston** (96.1%)
R1X3: Illinois vs Penn -> **Illinois** (91.0%)
R1X4: Nebraska vs Troy -> **Nebraska** (78.6%)
R1X5: Vanderbilt vs McNeese St -> **Vanderbilt** (79.0%)
R1X6: North Carolina vs VCU -> **North Carolina** (53.8%)
R1X7: St Mary's CA vs Texas A&M -> **St Mary's CA** (65.8%)
